# Heart Disease Prediction
**Models:** Logistic Regression · Random Forest · HistGradientBoosting  
**Goal:** High recall for *Presence* — catch every patient with heart disease  
**Split:** 70% Train · 15% Val (threshold tuning) · 15% Test (final evaluation)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, recall_score, precision_score,
    classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
    precision_recall_curve
)
import gc, warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Data

In [ ]:
def downcast(df):
    """Reduce memory by downcasting numeric types."""
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = df[col].astype('int32')
    return df

train_df = downcast(pd.read_csv('train.csv'))
test_df  = pd.read_csv('test.csv')   # no labels — used for final predictions only

print(f'Train: {train_df.shape[0]:,} rows, {train_df.shape[1]} columns')
print(f'Test:  {test_df.shape[0]:,} rows, {test_df.shape[1]} columns')
print(f'RAM (train): {train_df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print()
vc = train_df['Heart Disease'].value_counts()
print('Target distribution:')
print(vc.to_string())
print(f'\nPositive rate: {vc.get("Presence", 0) / len(train_df):.3f}')

## 2. Preprocessing

In [ ]:
FEATURES = [
    'Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
    'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
    'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium'
]

# Target: Presence = 1, Absence = 0
y_all = (train_df['Heart Disease'] == 'Presence').astype(int)
X_all = train_df[FEATURES].copy()

# Median imputation (robust to outliers)
imputer = SimpleImputer(strategy='median')
X_all = pd.DataFrame(imputer.fit_transform(X_all), columns=FEATURES)
X_test_sub = pd.DataFrame(imputer.transform(test_df[FEATURES]), columns=FEATURES)

print(f'Features: {len(FEATURES)}')
print(f'Missing values after imputation: {X_all.isnull().sum().sum()}')

## 3. Train / Val / Test Split

- **Train (70%)** — fit the models  
- **Val (15%)** — tune the decision threshold  
- **Test (15%)** — final evaluation, never seen during training

> `test.csv` has no labels and is only used to generate predictions at the end.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15 / 0.85,
    stratify=y_trainval,
    random_state=SEED
)

print(f'Train : {len(X_train):>7,}  ({len(X_train)/len(X_all):.0%})')
print(f'Val   : {len(X_val):>7,}  ({len(X_val)/len(X_all):.0%})')
print(f'Test  : {len(X_test):>7,}  ({len(X_test)/len(X_all):.0%})')

## 4. Threshold Tuning & Evaluation Helper

Instead of the default 0.5 threshold, we find the threshold that achieves **Recall ≥ 0.90** for *Presence* while maximising F1.  
This means we prefer to flag more patients as sick (more false positives) rather than miss anyone with heart disease.

In [ ]:
def find_best_threshold(y_true, y_proba, min_recall=0.90):
    """Return threshold with best F1 at recall >= min_recall."""
    prec, rec, thresholds = precision_recall_curve(y_true, y_proba)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    mask = rec[:-1] >= min_recall
    if mask.sum() == 0:
        return float(thresholds[np.argmax(f1[:-1])])
    return float(thresholds[np.where(mask)[0][np.argmax(f1[:-1][mask])]])


def evaluate_on_test(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te, min_recall=0.90):
    """Fit -> tune threshold on val -> evaluate on test."""
    model.fit(X_tr, y_tr)

    val_proba  = model.predict_proba(X_v)[:, 1]
    thr        = find_best_threshold(y_v, val_proba, min_recall)

    test_proba = model.predict_proba(X_te)[:, 1]
    test_pred  = (test_proba >= thr).astype(int)

    metrics = {
        'ROC-AUC'          : roc_auc_score(y_te, test_proba),
        'Accuracy'         : accuracy_score(y_te, test_pred),
        'Recall (Presence)': recall_score(y_te, test_pred),
        'Precision'        : precision_score(y_te, test_pred),
        'F1'               : f1_score(y_te, test_pred),
    }

    print(f'\n{"="*55}')
    print(f'  {name}  (threshold = {thr:.3f})')
    print(f'{"="*55}')
    print(f'  ROC-AUC            : {metrics["ROC-AUC"]:.4f}')
    print(f'  Accuracy           : {metrics["Accuracy"]:.4f}')
    print(f'  Recall  (Presence) : {metrics["Recall (Presence)"]:.4f}  <- patients caught')
    print(f'  Precision          : {metrics["Precision"]:.4f}')
    print(f'  F1-Score           : {metrics["F1"]:.4f}')
    print()
    print(classification_report(y_te, test_pred, target_names=['Absence', 'Presence']))

    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay.from_predictions(
        y_te, test_pred,
        display_labels=['Absence', 'Presence'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'{name} — Confusion Matrix (Test Set)')
    plt.tight_layout()
    plt.show()

    return model, thr, metrics

print('Helper functions ready.')

## 5. Model 1 — Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs',
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ))
])

lr_model, lr_thr, lr_metrics = evaluate_on_test(
    'Logistic Regression', lr_pipe,
    X_train, y_train, X_val, y_val, X_test, y_test
)

## 6. Model 2 — Random Forest

In [ ]:
rf_raw = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features='sqrt',
    max_samples=0.4,          # each tree sees 40% of data — saves RAM
    class_weight='balanced_subsample',
    n_jobs=4,
    random_state=SEED
)

rf_model, rf_thr, rf_metrics = evaluate_on_test(
    'Random Forest', rf_raw,
    X_train, y_train, X_val, y_val, X_test, y_test
)
gc.collect()

In [ ]:
# Feature importances
fi = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
fi.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.axvline(fi.mean(), color='red', linestyle='--', label='Mean')
ax.set_title('Random Forest — Feature Importances')
ax.set_xlabel('Importance')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Model 3 — HistGradientBoosting

In [ ]:
hgb_raw = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=0.1,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    class_weight='balanced',
    random_state=SEED
)

hgb_model, hgb_thr, hgb_metrics = evaluate_on_test(
    'HistGradientBoosting', hgb_raw,
    X_train, y_train, X_val, y_val, X_test, y_test
)
gc.collect()

## 8. Model Comparison — Test Set Results

In [ ]:
comparison = pd.DataFrame({
    'Logistic Regression' : lr_metrics,
    'Random Forest'       : rf_metrics,
    'HistGradientBoosting': hgb_metrics,
}).T.sort_values('ROC-AUC', ascending=False)

thresholds = {
    'Logistic Regression' : lr_thr,
    'Random Forest'       : rf_thr,
    'HistGradientBoosting': hgb_thr,
}
comparison['Threshold'] = comparison.index.map(thresholds)

print('=== RESULTS ON TEST SET (15% hold-out) ===')
cols = ['ROC-AUC', 'Accuracy', 'Recall (Presence)', 'Precision', 'F1', 'Threshold']
print(comparison[cols].to_string(float_format=lambda x: f'{x:.4f}'))
print()
print(f'Best ROC-AUC  : {comparison["ROC-AUC"].idxmax()}')
print(f'Best Recall   : {comparison["Recall (Presence)"].idxmax()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: all metrics
comparison[['ROC-AUC', 'Accuracy', 'Recall (Presence)', 'F1']].plot(
    kind='bar', ax=axes[0], edgecolor='black', width=0.7
)
axes[0].set_title('Model Comparison — Test Set', fontsize=13)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0.5, 1.02)
axes[0].axhline(0.9, color='red', linestyle='--', linewidth=1, label='Recall target 90%')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='x', rotation=15)

# Right: Recall vs Precision
x = np.arange(len(comparison))
w = 0.35
axes[1].bar(x - w/2, comparison['Recall (Presence)'], w,
            label='Recall (Presence)', color='tomato', edgecolor='black')
axes[1].bar(x + w/2, comparison['Precision'], w,
            label='Precision', color='steelblue', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison.index, rotation=15, ha='right')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0.5, 1.02)
axes[1].set_title('Recall vs. Precision — Presence Class', fontsize=13)
axes[1].axhline(0.9, color='red', linestyle='--', linewidth=1)
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. ROC & Precision-Recall Curves (Test Set)

**ROC curve** plots True Positive Rate vs. False Positive Rate across all thresholds.  
**AUC** (Area Under the Curve) summarises this into one number — the higher the better.  
The dots on the PR curve mark the chosen threshold for each model.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

model_info = [
    ('Logistic Regression',  lr_model,  lr_thr),
    ('Random Forest',        rf_model,  rf_thr),
    ('HistGradientBoosting', hgb_model, hgb_thr),
]

for name, model, thr in model_info:
    proba = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax1)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax2)

    pred = (proba >= thr).astype(int)
    ax2.scatter(
        recall_score(y_test, pred),
        precision_score(y_test, pred),
        s=90, zorder=5
    )

ax1.plot([0, 1], [0, 1], 'k--', label='Random baseline')
ax1.set_title('ROC Curves — Test Set')
ax1.legend(loc='lower right', fontsize=8)

ax2.set_title('Precision-Recall — Test Set\n(dots = chosen threshold)')
ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 10. Predict on test.csv & Save Submissions

The best model is retrained on **all** training data before predicting on `test.csv`.

In [ ]:
best_name = comparison['ROC-AUC'].idxmax()
best_model_map = {
    'Logistic Regression' : (lr_model,  lr_thr),
    'Random Forest'       : (rf_model,  rf_thr),
    'HistGradientBoosting': (hgb_model, hgb_thr),
}
best_m, best_thr = best_model_map[best_name]

print(f'Best model : {best_name}')
print(f'Threshold  : {best_thr:.3f}')
print(f'Retraining on all {len(X_all):,} training samples...')

best_m.fit(X_all, y_all)

sub_proba = best_m.predict_proba(X_test_sub)[:, 1]
sub_preds = (sub_proba >= best_thr).astype(int)

submission = pd.DataFrame({'id': test_df['id'], 'Heart Disease': sub_preds})
submission.to_csv('submission_best.csv', index=False)

print(f'Saved: submission_best.csv')
print(f'Predicted positive rate: {sub_preds.mean():.3f}')
submission.head(10)

In [ ]:
# Save predictions for all 3 models
all_models = [
    ('logistic_regression',    lr_model,  lr_thr),
    ('random_forest',          rf_model,  rf_thr),
    ('hist_gradient_boosting', hgb_model, hgb_thr),
]

for mname, m, thr in all_models:
    if mname != best_name.lower().replace(' ', '_'):
        m.fit(X_all, y_all)
    proba = m.predict_proba(X_test_sub)[:, 1]
    preds = (proba >= thr).astype(int)
    fname = f'submission_{mname}.csv'
    pd.DataFrame({'id': test_df['id'], 'Heart Disease': preds}).to_csv(fname, index=False)
    print(f'Saved {fname}  (positive rate: {preds.mean():.3f})')
    gc.collect()

## 11. Final Summary

In [ ]:
print('=' * 55)
print('  FINAL RESULTS — TEST SET (15% hold-out)')
print('=' * 55)
print(comparison[cols].to_string(float_format=lambda x: f'{x:.4f}'))
print()
print(f'  Best overall (ROC-AUC) : {comparison["ROC-AUC"].idxmax()}')
print(f'  Best recall            : {comparison["Recall (Presence)"].idxmax()}')
print()
print('  Submissions saved:')
print('    submission_best.csv                  <- best model')
print('    submission_logistic_regression.csv')
print('    submission_random_forest.csv')
print('    submission_hist_gradient_boosting.csv')